# Automated alignment of a laser on a camera with a piezo actuated mirror
awojdyla@lbl.gov, mamoonkhan@lbl.gov, Sept 2026

We want to control a piezo actuated mirror (thorlabs) to move the spot from a laser on a basler camera, and the use blop to center the beam based on camera statistics

First we want to connect to the camera and the piezo actuator, and wrap the python SDK in ophyd

## Connecting to instruments 

### Piezo mirror using the python SDK from Thorlabs

python API for thorlabs is here: 
"C:\Program Files (x86)\Thorlabs\MDT69XB\Sample\Thorlabs_MDT69XB_PythonSDK"

In [2]:
from piezo_control import PiezoController, PiezoError

# 1. Find connected devices
devices = PiezoController.list_devices()
if devices:
    sn, dtype = devices[0]
    try:
        with PiezoController(sn) as piezo:
            piezo.open()
            # Set X voltage to 10.0V
            piezo.set_voltage('X', 10.0)
            # Read all voltages
            x, y, z = piezo.get_xyz_voltages()
            print(f"Current Voltages - X: {x}, Y: {y}, Z: {z}")
    except PiezoError as e:
        print(f"Device Error: {e}")
else:
    print("No piezo devices found.")

Current Voltages - X: 0.0, Y: 0.0, Z: 0.0


In [3]:
piezo = PiezoController(sn)
piezo.open()
# Set X voltage to 10.0V
piezo.set_voltage('X', 10.0)
# Read all voltages
#x, y, z = piezo.get_xyz_voltages()
#print(f"Current Voltages - X: {x}, Y: {y}, Z: {z}")

In [4]:
piezo.get_voltage('X')

10.12

In [5]:
piezo.set_voltage('X', 0.0)

In [6]:
piezo.close()

### Using pylablib with MDT694B (read-only checks)

Both COM4 and COM5 identify as MDT694B interfaces. Their earlier `get_id()` queries returned firmware 1.10, but `get_voltage_range()` stalled on the `%` command on **both** ports. That query is implemented for the MDT69xA protocol and is not reliable with this MDT694B setup. The examples below only query firmware identity; they do not change voltage. Close other software using these COM ports before retrying. If voltage readback is needed, test it separately with a short timeout or use the MDT69XB SDK in `piezo_control.py`. Do not interpret the old `KeyboardInterrupt` trace as a failure of `close()`: the traceback points to `get_voltage_range()` inside the `try` block.

In [8]:
# from pylablib.devices import Thorlabs

# # Read-only check; do not query the voltage range (the '%' command hung).
# piezo_1 = Thorlabs.MDT69xA("COM4")
# try:
#     print('COM4:', piezo_1.get_id())
# finally:
#     piezo_1.close()

## New Thorlabs Kinesis KCube DC controller (read-only)

Use the pip-installable `pylablib` package, already used for Kinesis motors elsewhere in this repository. Install it into this notebook's kernel with `%pip install pylablib` if needed, then restart the kernel. The new KCube is separate from the MDT69XB piezo above: `piezo_control.py` cannot operate it. The model may be KDC101 (rather than KCD101); confirm the controller and attached stage labels before attempting any motion. USB drivers may still be required; pip does not install hardware drivers. Discovery does not open or move the stage. On this Windows host, an APT USB Device with serial `27006231` was present and `pylablib 1.4.5` discovered it from the Miniconda Python interpreter. If the notebook returns an empty list, compare the printed Python executable and package version with that interpreter, close the Thorlabs Kinesis GUI and retry, and confirm the controller appears in Windows Device Manager.

In [ ]:
import sys
import pylablib
from pylablib.devices import Thorlabs

print('Notebook Python:', sys.executable)
print('pylablib version:', pylablib.__version__)
# Discovery only; check the serial against the controller label before opening it.
kinesis_devices = Thorlabs.list_kinesis_devices()
print('Available Kinesis devices (serial, description):', kinesis_devices)
if not kinesis_devices:
    print('No devices detected. Check the USB driver in Device Manager and close the Kinesis GUI before retrying.')

## Read the KCube position and move 1 mm forward

Confirm that serial `27006231` is the intended controller, identify the attached *linear* stage, and check that it is homed and has at least 1 mm of unobstructed travel in the positive direction. Close other software controlling it. The code uses `scale='stage'`, for which a detected linear stage reports positions in **meters**; it refuses to move if the stage or meter units are not detected. First run with `allow_move = False` to inspect the current and proposed positions; only set it to `True` after checking the stage's physical travel limits. This code does not home the stage.

In [10]:
from pylablib.devices import Thorlabs

kcube_serial = '27006231'  # Verify against the controller label.
allow_move = True  # Change to True only after checking homing, limits, and clearance.

if kcube_serial not in [str(device[0]) for device in Thorlabs.list_kinesis_devices()]:
    raise RuntimeError(f'KCube {kcube_serial} was not found')

motor = Thorlabs.KinesisMotor(kcube_serial, scale='stage')
try:
    stage = motor.get_stage()
    units = motor.get_scale_units()
    if not stage or units != 'm':
        raise RuntimeError(f'Cannot confirm a linear stage with meter units (stage={stage!r}, units={units!r}); do not move')
    current_m = motor.get_position()
    target_m = current_m + 0.001  # 1 mm = 0.001 m.
    print(f'Stage: {stage}; current: {current_m * 1000:.4f} mm; proposed: {target_m * 1000:.4f} mm')
    if allow_move:
        motor.move_to(target_m)
        motor.wait_move(timeout=30)
        print(f'Position after move: {motor.get_position() * 1000:.4f} mm')
    else:
        print('No motion commanded. Check travel limits and clearance, then set allow_move = True to move.')
finally:
    motor.close()

Stage: MTS50-Z8; current: 25.8389 mm; proposed: 26.8389 mm
Position after move: 26.8389 mm


## Reading the camera

In [ ]:
from pypylon import pylon

# Select the discovered GigE camera by its IP address.
camera_ip = "192.168.10.212"
factory = pylon.TlFactory.GetInstance()
device_info = next(
    (device for device in factory.EnumerateDevices()
     if hasattr(device, "GetIpAddress") and device.GetIpAddress() == camera_ip),
    None,
)
if device_info is None:
    raise RuntimeError(f"No accessible Basler camera found at {camera_ip}")

camera = pylon.InstantCamera(factory.CreateDevice(device_info))
try:
    camera.Open()
    grab_result = camera.GrabOne(5000)
    try:
        if not grab_result.GrabSucceeded():
            raise RuntimeError(
                f"Camera grab failed: {grab_result.GetErrorCode()} "
                f"{grab_result.GetErrorDescription()}"
            )
        converter = pylon.ImageFormatConverter()
        converter.OutputPixelFormat = pylon.PixelType_Mono8
        img_array = converter.Convert(grab_result).GetArray().copy()
    finally:
        grab_result.Release()
finally:
    camera.Close()